In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler

In [3]:
df = pd.read_csv("https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv")
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.drop(columns=['id','Unnamed: 32'],inplace=True)

In [5]:
X_train,X_test,y_train,y_test = train_test_split(df.iloc[:,1:],df.iloc[:,0],test_size=0.2)

In [6]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [8]:
X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

In [9]:
from torch.utils.data import DataLoader,Dataset

In [10]:
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return self.features.shape[0]
    
    def __getitem__(self, idx):
        return self.features[idx],self.labels[idx]

In [11]:
train_dataset = CustomDataset(X_train_tensor,y_train_tensor)
test_dataset = CustomDataset(X_test_tensor,y_test_tensor)

In [20]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=True)

In [21]:
import torch.nn as nn

In [22]:
class SimpleNN(nn.Module):
    def __init__(self,num_features):
        super().__init__()
        self.linear = nn.Linear(num_features,1)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self,features):
        out = self.linear(features)
        out = self.sigmoid(out)
        return out
    

In [23]:
learning_rate = 0.1
epochs = 25

In [24]:
model = SimpleNN(X_train_tensor.shape[1])

optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate)

loss_fun = nn.BCELoss()

Training pipeline

In [25]:
for epoch in range(epochs):
    for batch_features,batch_labels in train_loader:
        y_pred = model(batch_features)

        loss = loss_fun(y_pred,batch_labels.view(-1,1))

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

    print(f"Epoch:{epoch+1}, Loss:{loss.item()}")

Epoch:1, Loss:0.3124757707118988
Epoch:2, Loss:0.21053561568260193
Epoch:3, Loss:0.10157321393489838
Epoch:4, Loss:0.0792979821562767
Epoch:5, Loss:0.1374451071023941
Epoch:6, Loss:0.025106241926550865
Epoch:7, Loss:0.15283386409282684
Epoch:8, Loss:0.10868920385837555
Epoch:9, Loss:0.1309516876935959
Epoch:10, Loss:0.18768851459026337
Epoch:11, Loss:0.007880317978560925
Epoch:12, Loss:0.021583762019872665
Epoch:13, Loss:0.049590129405260086
Epoch:14, Loss:0.137458935379982
Epoch:15, Loss:0.6623594164848328
Epoch:16, Loss:0.009359325282275677
Epoch:17, Loss:0.009225637651979923
Epoch:18, Loss:0.0640600323677063
Epoch:19, Loss:0.01476198248565197
Epoch:20, Loss:0.03516311198472977
Epoch:21, Loss:0.34926244616508484
Epoch:22, Loss:0.11301258951425552
Epoch:23, Loss:0.09954027831554413
Epoch:24, Loss:0.008736749179661274
Epoch:25, Loss:0.005525160115212202


In [26]:
model.eval() # set model to evaluation mode
accuracy_list = []
with torch.no_grad():
    for batch_features,batch_labels in test_loader:
        #frwd pass
        y_pred = model(batch_features)
        y_pred = (y_pred > 0.8).float()# prob to binary pred

        batch_accuracy = (y_pred.view(-1) == batch_labels).float().mean().item()# accuracy for current batch
        accuracy_list.append(batch_accuracy)

overall_accuracy = sum(accuracy_list)/len(accuracy_list)
print(f"Accuracy: {overall_accuracy:.4f}")

Accuracy: 0.9627
